In [3]:
import json
import os

import pandas as pd
from openai import OpenAI

In [43]:
client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY")
)

EVALUATOR_MODEL = "gpt-5.4-mini"

print("OpenAI client ready.")

OpenAIError: Missing credentials. Please pass an `api_key`, `workload_identity`, `admin_api_key`, or set the `OPENAI_API_KEY` or `OPENAI_ADMIN_KEY` environment variable.

In [20]:
with open("rag_summaries.json", "r") as f:
    rag_summaries = json.load(f)

print("Loaded summaries:", len(rag_summaries))
print(rag_summaries.keys())

Loaded summaries: 9
dict_keys(['Whole + Gemini', 'Whole + BGE', 'Whole + MedCPT', 'Fixed + Gemini', 'Fixed + BGE', 'Fixed + MedCPT', 'Section + Gemini', 'Section + BGE', 'Section + MedCPT'])


In [21]:
temporal_events_df = pd.read_csv(
    "temporal_candidate_events.csv"
)

In [40]:
TEMPORAL_CONSOLIDATION_PROMPT = """
You are consolidating clinical events for longitudinal temporal evaluation.

The source clinical notes have already been ordered by creation_timestamp.
The event's source note index therefore represents its authoritative
chronological position.

For each CURRENT candidate event, determine whether it represents:

KEEP:
- a genuinely new clinical event
- a new investigation, treatment, diagnosis, referral, or disposition
- a meaningful change in clinical state
- a new value/finding that represents clinical progression
- a treatment being started, stopped, changed, or meaningfully continued

REMOVE:
- the same clinical event already represented in EARLIER events
- historical information merely restated
- duplicated findings/results with no new clinical change
- administrative/non-clinically meaningful information

Important:
Do NOT remove genuine progression simply because it concerns the same
clinical concept.

Example:
"SpO2 89% on room air"
"SpO2 improved to 92% after oxygen"
"SpO2 later 94% on room air"
are separate meaningful states and should all be kept.

But repeated statements of the same NT-proBNP result of 4500 pg/mL,
without a new measurement or change, should not create multiple events.

Do not use dates written inside event text to establish chronology.
Use the supplied source note indices only.

Return valid JSON only:

{
  "decisions": [
    {
      "candidate_id": 0,
      "decision": "KEEP" or "REMOVE",
      "reason": "brief reason"
    }
  ]
}
"""

In [41]:
def consolidate_note_events(current_events, earlier_events):

    # Give each current candidate an ID so we can map
    # Nemotron's decision back to the original event.
    current_candidates = [
        {
            "candidate_id": i,
            "event": event
        }
        for i, event in enumerate(current_events)
    ]

    user_message = f"""
EARLIER CONSOLIDATED EVENTS:
{json.dumps(earlier_events, indent=2)}

CURRENT CANDIDATE EVENTS:
{json.dumps(current_candidates, indent=2)}
"""

    response = openrouter_client.chat.completions.create(
        model=EVALUATOR_MODEL,
        messages=[
            {
                "role": "system",
                "content": TEMPORAL_CONSOLIDATION_PROMPT
            },
            {
                "role": "user",
                "content": user_message
            }
        ],
        temperature=0,
        response_format={"type": "json_object"}
    )

    # Get the model output safely.
    raw_output = response.choices[0].message.content

    if raw_output is None or not raw_output.strip():
        raise ValueError(
            "Nemotron returned an empty response."
        )

    # Remove Markdown fences if the model returns ```json ... ```
    raw_output = raw_output.strip()

    if raw_output.startswith("```"):
        raw_output = raw_output.replace("```json", "")
        raw_output = raw_output.replace("```", "")
        raw_output = raw_output.strip()

    result = json.loads(raw_output)

    return result["decisions"]

In [42]:
test_consolidated_events = []
test_decisions = []

for note_index in [0, 1, 2]:

    current_df = temporal_events_df[
        temporal_events_df["source_note_index"] == note_index
    ]

    current_events = current_df["event"].tolist()

    decisions = consolidate_note_events(
        current_events=current_events,
        earlier_events=[
            item["event"]
            for item in test_consolidated_events
        ]
    )

    for decision in decisions:

        candidate_id = decision["candidate_id"]
        event_text = current_events[candidate_id]

        record = {
            "source_note_index": note_index,
            "event": event_text,
            "decision": decision["decision"],
            "reason": decision["reason"]
        }

        test_decisions.append(record)

        if decision["decision"] == "KEEP":
            test_consolidated_events.append({
                "source_note_index": note_index,
                "event": event_text
            })


for item in test_decisions:
    print(
        f"NOTE {item['source_note_index']} | "
        f"{item['decision']} | "
        f"{item['event']}"
    )
    print("   Reason:", item["reason"])

RateLimitError: Error code: 429 - {'error': {'message': 'Provider returned error', 'code': 429, 'metadata': {'raw': 'google/gemma-4-26b-a4b-it:free is temporarily rate-limited upstream. Please retry shortly, or add your own key to accumulate your rate limits: https://openrouter.ai/settings/integrations', 'provider_name': 'Google AI Studio', 'is_byok': False, 'provider_error_code': '429', 'limit_source': 'upstream_provider_shared_pool', 'remedy_hint': 'Retry shortly, add your own provider key (https://openrouter.ai/settings/integrations), or route to another provider with provider routing: https://openrouter.ai/docs/features/provider-routing'}}, 'user_id': 'user_3IpIe2Ob9ZShGDy7LNmio3afPGP'}

In [38]:
import requests

models = requests.get(
    "https://openrouter.ai/api/v1/models"
).json()["data"]

for model in models:
    name = model["name"].lower()

    if "gemma 4" in name:
        print(
            "ID:", model["id"],
            "\nName:", model["name"],
            "\nPrice:", model["pricing"],
            "\nSupported:", model["supported_parameters"]
        )

ID: google/gemma-4-26b-a4b-it 
Name: Google: Gemma 4 26B A4B  
Price: {'prompt': '0.00000007', 'completion': '0.00000034'} 
Supported: ['frequency_penalty', 'include_reasoning', 'logit_bias', 'logprobs', 'max_tokens', 'min_p', 'presence_penalty', 'reasoning', 'repetition_penalty', 'response_format', 'seed', 'stop', 'structured_outputs', 'temperature', 'tool_choice', 'tools', 'top_k', 'top_logprobs', 'top_p']
ID: google/gemma-4-26b-a4b-it:free 
Name: Google: Gemma 4 26B A4B  (free) 
Price: {'prompt': '0', 'completion': '0'} 
Supported: ['include_reasoning', 'max_tokens', 'reasoning', 'response_format', 'seed', 'temperature', 'tool_choice', 'tools', 'top_p']
ID: google/gemma-4-31b-it 
Name: Google: Gemma 4 31B 
Price: {'prompt': '0.00000009', 'completion': '0.00000034', 'input_cache_read': '0.00000005'} 
Supported: ['frequency_penalty', 'include_reasoning', 'logit_bias', 'logprobs', 'max_tokens', 'min_p', 'presence_penalty', 'reasoning', 'repetition_penalty', 'response_format', 'seed', '